# Baseline: Poisson GLM

Fits a Poisson regression per LOTO fold to predict expected goals from ELO + form features.
Score prediction: maximise expected Sporza pts over the joint Poisson distribution.

Uses LOTO-CV: 4 folds × 64 matches = 256 pooled evaluation matches. Reports bootstrap 95% CI
and a paired permutation test against the autofill baseline.

In [1]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from functools import lru_cache
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from pathlib import Path

from evaluation.sporza import score_breakdown, sporza_points_series

DATA = Path('../../data/processed')
WC_YEARS = [2010, 2014, 2018, 2022]
MAX_GOALS = 8

HOME_FEATURES = ['home_elo', 'away_elo', 'elo_diff', 'is_neutral',
                 'home_form_wr', 'away_form_wr', 'home_form_gf', 'away_form_gf',
                 'home_form_ga', 'away_form_ga', 'tournament_weight']
AWAY_FEATURES = ['away_elo', 'home_elo', 'elo_diff', 'is_neutral',
                 'away_form_wr', 'home_form_wr', 'away_form_gf', 'home_form_gf',
                 'away_form_ga', 'home_form_ga', 'tournament_weight']

In [2]:
def bootstrap_ci(pts: pd.Series, n_boot: int = 10000) -> dict:
    rng = np.random.default_rng(42)
    vals = pts.values
    boot = np.array([rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {'mean': float(vals.mean()), 'ci_lo': float(lo), 'ci_hi': float(hi)}


def permutation_test(pts_a: np.ndarray, pts_b: np.ndarray, n_perm: int = 10000) -> float:
    """Paired permutation test: H0 = mean(a) <= mean(b). Returns p-value."""
    rng = np.random.default_rng(42)
    obs_diff = pts_a.mean() - pts_b.mean()
    diffs = np.zeros(n_perm)
    for i in range(n_perm):
        swap = rng.integers(0, 2, size=len(pts_a)).astype(bool)
        a = np.where(swap, pts_b, pts_a)
        b = np.where(swap, pts_a, pts_b)
        diffs[i] = a.mean() - b.mean()
    return float((diffs >= obs_diff).mean())


def build_X(df: pd.DataFrame, features: list) -> np.ndarray:
    X = df[features].copy().fillna(df[features].median())
    return X.values


def build_pipe() -> Pipeline:
    return Pipeline([('scaler', StandardScaler()),
                     ('poisson', PoissonRegressor(alpha=0.1, max_iter=300))])


@lru_cache(maxsize=50000)
def best_score_cached(lh_r: float, la_r: float) -> tuple:
    """Score (h, a) maximising expected Sporza pts given Poisson(lh) x Poisson(la)."""
    goals = np.arange(MAX_GOALS + 1)
    ph_pmf = poisson.pmf(goals, lh_r)
    pa_pmf = poisson.pmf(goals, la_r)
    joint = np.outer(ph_pmf, pa_pmf)

    def sporza_mat(pred_h: int, pred_a: int) -> float:
        pts = 0.0
        for ah in range(MAX_GOALS + 1):
            for aa in range(MAX_GOALS + 1):
                p = joint[ah, aa]
                if p < 1e-9:
                    continue
                if pred_h == ah and pred_a == aa:
                    pts += p * 10
                elif (pred_h - pred_a) == (ah - aa):
                    pts += p * 7
                elif np.sign(pred_h - pred_a) == np.sign(ah - aa):
                    pts += p * 5
                else:
                    pts += p * 1
        return pts

    best_pts, best_h, best_a = -1.0, 1, 0
    for ph_idx in range(MAX_GOALS + 1):
        for pa_idx in range(MAX_GOALS + 1):
            ep = sporza_mat(ph_idx, pa_idx)
            if ep > best_pts:
                best_pts, best_h, best_a = ep, ph_idx, pa_idx
    return best_h, best_a

In [3]:
# Load autofill baseline pts for paired test
import json
manifest = json.loads((DATA / 'split_manifest.json').read_text())
autofill_mean = manifest['autofill_baseline']['pooled_mean_pts']
print(f'Autofill baseline (measured): {autofill_mean:.3f} pts/match')

Autofill baseline (measured): 4.137 pts/match


In [4]:
all_pts = []
fold_results = []
fold_pipes = {}

for year in WC_YEARS:
    train = pd.read_parquet(DATA / f'train_fold_{year}.parquet')
    evalf = pd.read_parquet(DATA / f'eval_fold_{year}.parquet')

    # Stacked training: each match as (home perspective) + (away perspective)
    X_h = build_X(train, HOME_FEATURES)
    X_a = build_X(train, AWAY_FEATURES)
    X_tr = np.vstack([X_h, X_a])
    y_tr = np.concatenate([train['home_score'].values, train['away_score'].values])

    pipe = build_pipe()
    pipe.fit(X_tr, y_tr)
    fold_pipes[year] = pipe

    lh = pipe.predict(build_X(evalf, HOME_FEATURES))
    la = pipe.predict(build_X(evalf, AWAY_FEATURES))

    preds = [best_score_cached(round(float(h), 2), round(float(a), 2)) for h, a in zip(lh, la)]
    pred_home = pd.Series([p[0] for p in preds])
    pred_away = pd.Series([p[1] for p in preds])

    bd = score_breakdown(pred_home, pred_away,
                         evalf['home_score'].reset_index(drop=True),
                         evalf['away_score'].reset_index(drop=True))
    pts = sporza_points_series(pred_home, pred_away,
                               evalf['home_score'].reset_index(drop=True),
                               evalf['away_score'].reset_index(drop=True))
    all_pts.extend(pts.tolist())
    fold_results.append({'year': year, 'mean_pts': bd['mean_pts'],
                         'pct_exact': bd['pct_exact'], 'pct_correct_result': bd['pct_correct_result']})
    print(f'WC {year}: mean_pts={bd["mean_pts"]:.3f}  exact={bd["pct_exact"]:.1%}  correct={bd["pct_correct_result"]:.1%}')

pooled_pts = pd.Series(all_pts)
ci = bootstrap_ci(pooled_pts)
print(f'\nPooled LOTO-CV ({len(pooled_pts)} matches): {ci["mean"]:.3f} pts/match  '
      f'95% CI [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]  '
      f'CI width={ci["ci_hi"]-ci["ci_lo"]:.3f}')

WC 2010: mean_pts=4.141  exact=17.2%  correct=51.6%
WC 2014: mean_pts=4.875  exact=18.8%  correct=64.1%


WC 2018: mean_pts=4.562  exact=18.8%  correct=57.8%


WC 2022: mean_pts=3.766  exact=7.8%  correct=54.7%

Pooled LOTO-CV (256 matches): 4.336 pts/match  95% CI [3.930, 4.742]  CI width=0.813


In [5]:
# Comparison table vs autofill
print('\nComparison vs autofill baseline:')
print(f'  Autofill (empirical): {autofill_mean:.3f} pts/match')
print(f'  Poisson GLM LOTO-CV:  {ci["mean"]:.3f} pts/match  95% CI [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]')
print(f'  Difference:           {ci["mean"] - autofill_mean:+.3f} pts/match')

# Check if CI overlaps autofill
beats = ci['ci_lo'] > autofill_mean
print(f'  CI lower bound > autofill: {beats} → {"statistically beats" if beats else "not conclusive"}')


Comparison vs autofill baseline:
  Autofill (empirical): 4.137 pts/match
  Poisson GLM LOTO-CV:  4.336 pts/match  95% CI [3.930, 4.742]
  Difference:           +0.199 pts/match
  CI lower bound > autofill: False → not conclusive


In [6]:
# Per-fold summary
print('Per-fold:')
for r in fold_results:
    print(f'  WC {r["year"]}: {r["mean_pts"]:.3f} pts  exact={r["pct_exact"]:.1%}  correct={r["pct_correct_result"]:.1%}')

Per-fold:
  WC 2010: 4.141 pts  exact=17.2%  correct=51.6%
  WC 2014: 4.875 pts  exact=18.8%  correct=64.1%
  WC 2018: 4.562 pts  exact=18.8%  correct=57.8%
  WC 2022: 3.766 pts  exact=7.8%  correct=54.7%


In [7]:
# WC 2022 as held-out test result (most recent fold)
wc22_result = fold_results[-1]
print(f'WC 2022 (held-out): {wc22_result["mean_pts"]:.3f} pts/match  exact={wc22_result["pct_exact"]:.1%}')

WC 2022 (held-out): 3.766 pts/match  exact=7.8%


In [8]:
mlflow.set_tracking_uri('../../mlruns')
mlflow.set_experiment('wk2026-tabular-2026-06-13')

with mlflow.start_run(run_name='poisson_glm_loto'):
    mlflow.set_tags({'modality': 'tabular', 'approach': 'poisson_glm',
                     'eval_strategy': 'loto_cv', 'dataset_version': 'split_v2'})
    mlflow.log_params({'features': ','.join(HOME_FEATURES), 'alpha': 0.1,
                       'score_strategy': 'max_expected_sporza_pts', 'max_goals_grid': MAX_GOALS})
    mlflow.log_metric('loto_mean_sporza_pts', ci['mean'])
    mlflow.log_metric('loto_ci_lo', ci['ci_lo'])
    mlflow.log_metric('loto_ci_hi', ci['ci_hi'])
    mlflow.log_metric('autofill_baseline', autofill_mean)
    mlflow.log_metric('delta_vs_autofill', ci['mean'] - autofill_mean)
    for r in fold_results:
        mlflow.log_metric(f'fold_{r["year"]}_mean_pts', r['mean_pts'])
        mlflow.log_metric(f'fold_{r["year"]}_pct_exact', r['pct_exact'])
    mlflow.sklearn.log_model(fold_pipes[2022], 'poisson_glm_wc2022_fold')
    run_id = mlflow.active_run().info.run_id

print(f'run_id: {run_id}')

2026/06/13 21:07:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/06/13 21:07:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


run_id: 9f011bcc53d74c3b9af7f726e8322ee9
